In [1]:
import os

In [2]:
%pwd

'c:\\Users\\Lenovo\\Desktop\\End-to-End-Project\\kidney_disease_classification_end-to-end-DL\\research'

In [9]:
os.chdir("../")

In [10]:
%pwd

'c:\\Users\\Lenovo\\Desktop\\End-to-End-Project\\kidney_disease_classification_end-to-end-DL'

In [11]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class DataIngestionConfig:
    root_dir: Path
    source_URL: str
    local_data_file: Path
    unzip_dir: Path

In [12]:
from cnnClassifier.constants import *
from cnnClassifier.utils.common import read_yaml, create_directories

In [13]:
class ConfigurationManager:
    def __init__(
            self,
            config_file_path = CONFIG_FILE_PATH,
            params_file_path = PARAMS_FILE_PATH
            ):
        self.config = read_yaml(config_file_path)
        self.params = read_yaml(params_file_path)
        create_directories([self.config.artifacts_root])

    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config = self.config.data_ingestion

        create_directories([config.root_dir])

        data_ingestion_config = DataIngestionConfig(
            root_dir = config.root_dir,
            source_URL = config.source_URL,
            local_data_file = config.local_data_file,
            unzip_dir = config.unzip_dir
        )

        return data_ingestion_config

In [14]:
import os
import zipfile
import gdown
from cnnClassifier import logger
from cnnClassifier.utils.common import get_size

In [27]:
class DataIngestion:
    def __init__(self, config: DataIngestionConfig):
        self.config = config

    def download_file(self):
        try:
            dataset_url = self.config.source_URL
            zip_download_dir = self.config.local_data_file
            os.makedirs(os.path.dirname(zip_download_dir), exist_ok=True)  # ← use config path
            logger.info(f"downloading file from :[{dataset_url}] into :[{zip_download_dir}]")

            file_id = dataset_url.split("/")[-2]
            prefix = "https://drive.google.com/uc?export=download&id="
            gdown.download(prefix + file_id, str(zip_download_dir))

            logger.info(f"file downloaded from :[{dataset_url}] into :[{zip_download_dir}]")

        except Exception as e:
            raise e

    def extract_zip_file(self):
        unzip_path = self.config.unzip_dir
        os.makedirs(unzip_path, exist_ok=True)
        with zipfile.ZipFile(self.config.local_data_file, 'r') as zip_ref:
            zip_ref.extractall(unzip_path)
        logger.info(f"Extracted zip file into: {unzip_path}")

In [28]:
try:
    config = ConfigurationManager()
    data_ingestion_config = config.get_data_ingestion_config()
    print(data_ingestion_config)

    data_ingestion = DataIngestion(config=data_ingestion_config)
    data_ingestion.download_file()
    data_ingestion.extract_zip_file()

except Exception as e:
    raise e

[2026-03-13 00:57:01,061: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-03-13 00:57:01,066: INFO: common: yaml file: params.yaml loaded successfully]
[2026-03-13 00:57:01,071: INFO: common: created directory at: artifacts]
[2026-03-13 00:57:01,075: INFO: common: created directory at: artifacts/data_ingestion]
DataIngestionConfig(root_dir='artifacts/data_ingestion', source_URL='https://drive.google.com/file/d/1vlhZ5c7abUKF8xXERIw6m9Te8fW7ohw3/view?usp=sharing', local_data_file='artifacts/data_ingestion/data.zip', unzip_dir='artifacts/data_ingestion')
[2026-03-13 00:57:01,079: INFO: 2100078176: downloading file from :[https://drive.google.com/file/d/1vlhZ5c7abUKF8xXERIw6m9Te8fW7ohw3/view?usp=sharing] into :[artifacts/data_ingestion/data.zip]]


Downloading...
From (original): https://drive.google.com/uc?export=download&id=1vlhZ5c7abUKF8xXERIw6m9Te8fW7ohw3
From (redirected): https://drive.google.com/uc?export=download&id=1vlhZ5c7abUKF8xXERIw6m9Te8fW7ohw3&confirm=t&uuid=673cfd36-2647-42ce-9ec3-0e9528cb7240
To: c:\Users\Lenovo\Desktop\End-to-End-Project\kidney_disease_classification_end-to-end-DL\artifacts\data_ingestion\data.zip
100%|██████████| 57.7M/57.7M [00:11<00:00, 4.81MB/s]

[2026-03-13 00:57:16,390: INFO: 2100078176: file downloaded from :[https://drive.google.com/file/d/1vlhZ5c7abUKF8xXERIw6m9Te8fW7ohw3/view?usp=sharing] into :[artifacts/data_ingestion/data.zip]]


[2026-03-13 00:57:17,917: INFO: 2100078176: Extracted zip file into: artifacts/data_ingestion]
